# B2.5 · Feasibility filtering, reachability and dead code

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *AI for Security*

Builds on **[B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**.

| | |
|---|---|
| Tools used | CodeQL, tree-sitter, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Build a call graph from entry points and partition findings into reachable, unreachable and unknown.

**Why a security engineer needs it.** A finding in dead code costs the same to triage as one on the login path. The control it builds is: stage 10: decide whether an external caller can actually reach the sink before anyone is paged.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The finding is real and the code is dead — a true positive about the code and a false positive about the risk. Telling those apart needs a call graph, a call graph needs the syntax tree, and the tree's own blind spot is the third bucket everyone collapses into the second.

> **At CyberTravels.** The finding is real and nothing in CyberTravels' booking service calls the function it landed in. The syntax tree can prove that for three of them, and for the nightly ledger job it cannot — which is the finding, not a gap in the report.

## 2 · The framework

```
   grep says            ast says

   "report" appears     FunctionDef report    <- a node
   in 14 files          Call report(...)      <- an edge, owned by the
                                                 function it sits inside

   nodes + edges, walked from the entry points:

     @route handler --> _query           REACHABLE, keeps its severity

     legacy_export      (no caller)      UNREACHABLE
     audit_line         (no caller)      -> true about the code
     debug_dump         (no caller)      -> FALSE about the risk

     jobs/runner.py:  handler = getattr(HANDLERS, name)
                      return handler(arg)

     run_job                             UNKNOWN
     nightly_reconcile                   -> not dead.
     _settle                                undecided.

   the third bucket is not a rounding error: _settle runs a SQL
   update, and a two-bucket pipeline reports it as clean
```

**Stage 10 — Feasibility filtering.** The last stage of Phase 3, and the one
that decides whether anyone gets paged.

A verified finding is a real bug in the code. It is not necessarily a real risk,
because the code may be unreachable: dead code, a test fixture, an internal
function no external caller can drive, a branch behind a feature flag that has
been off for two years.

Triaging an unreachable finding costs exactly as much as triaging one on the
login path, and there are usually far more of them. So this stage partitions
findings into three buckets — and the third bucket is the honest one:

- **reachable** — a path exists from an untrusted entry point to the sink,
- **unreachable** — no path exists,
- **unknown** — the analysis cannot decide, usually because of dynamic dispatch,
  reflection, or a framework that wires callers at runtime.

Reporting `unknown` as `unreachable` is how a pipeline quietly drops real bugs.

### Dead code, and the two different claims a finding makes

The largest single class in that unreachable bucket is **dead code**, and it is
worth being precise about what is wrong with such a finding, because teams act
on the wrong half.

The finding is a **true positive about the code**. The concatenation is there,
the sink is real, and any reviewer who opens the file will agree. It is a
**false positive about the risk**, because nothing untrusted reaches it. Two
different claims, and only the second one is wrong.

### Deciding which is which needs the AST

You cannot answer it with grep. `def report`, `report(` and `# report` are the
same string to a regex, and a function called `run` appears in every file you
own. The question — *which functions call this one* — is about structure, so it
needs the **abstract syntax tree**.

Parsing gives you exactly the two node types the question needs:

| AST node | what it gives you |
|---|---|
| `FunctionDef` | every function that exists, with its real nesting and its decorators |
| `Call` | every invocation, attributable to the function it sits inside |

Nodes and edges. Walk from the entry points and everything you reach is
reachable; everything you do not is dead **or** undecided.

The resolver should be deliberately naive. `Call.func` is a `Name` for `f()`
and an `Attribute` for `obj.f()`; take `.id` or `.attr` and accept that two
methods with the same name merge. That over-reports reachability, which is the
safe error. A cleverer resolver that guesses wrong marks a live function dead,
and that error is silent.

### And the AST is honest about what it cannot see

This is the more useful half, and it is where the third bucket comes from. The
AST resolves a literal call. It cannot resolve

```python
handler = getattr(HANDLERS, name)   # the callee is a runtime value
return handler(arg)
```

nor a dispatch dictionary, nor a handler a framework registers by decorator at
import time. None of those are unreachable — they are **undecided**, and the
rule that follows is the one that keeps the analysis honest: if a module
contains a call the AST could not follow, *every* unreached function in that
module is `unknown`, not `unreachable`.

Deleting is the right resolution for the genuinely dead ones, and it is the only
one that cannot rot: a suppression is keyed to a file, a line and a rule, and
none of those change when somebody wires the function back up.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Where it breaks — collapsing `unknown` into `unreachable`

The tempting simplification. It makes the queue shorter and it is how real bugs get dropped, because dynamic dispatch is exactly where framework-wired handlers live.

## 4 · Phase 3 as a skill — and the counts that police it

Stages 7 to 10 only ever *shrink* the list. That is a property worth enforcing rather than trusting, so the skill's contract carries a `counts` object and the rule that it must never increase.

A pipeline whose `verified` count exceeds its `deduped` count has invented findings somewhere after the audit stage — and that is far easier to do by accident than it sounds, because a verification step that expands one finding per code path looks perfectly reasonable from the inside.

### The skill — [`skills/appsec/appsec-vuln-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/SKILL.md)

```yaml
name: appsec-vuln-audit
description: >-
  Audit code for vulnerabilities against a threat model, then deduplicate,
  verify in context, and filter to what is actually reachable. Use when asked
  to review code for security bugs, run or interpret SAST, check whether a
  finding is a false positive, or reduce a noisy findings list to the ones
  worth a human's time.
allowed-tools: Read, Grep, Glob, Bash
```

# AppSec pipeline · Phase 3 — Analysis and filtering

Covers **stages 7–10**. This is where findings are produced — and, more
importantly, where most of them are thrown away.

The hard problem in application security is not finding candidate defects. It
is that a scanner emits hundreds and a human can act on ten. Every stage after
7 exists to shrink the list without losing the true positives.

## When to use this

When you have a threat model and a budget, or when handed a raw findings file
that nobody trusts. Stages 8–10 work on any findings list, including one from a
third-party scanner.

## Inputs

| Input | Required | Notes |
|---|---|---|
| `threat_model` + `plan.selected` | preferred | from appsec-threat-model |
| Source worktree | yes | verification needs the code, not just the finding |
| Existing findings | optional | run stages 8–10 alone to clean a noisy list |

## Procedure

**Stage 7 — Vulnerability auditing.** For each selected threat, examine the
path from entry to sink and decide whether the weakness is actually present.
Record for each finding: `cwe`, `file`, `line`, `unit`, the **evidence** (the
specific expression that is unsafe), and the **sanitiser** you looked for and
did not find. A finding that cannot name what was missing is a guess.

Three generations of analysis, and they are complementary, not competing:
grep-class pattern matching (fast, no dataflow), taint analysis (dataflow, no
semantics), and model-assisted review (semantics, no guarantees). Use the
cheapest one that can answer the question, and never let the third overrule the
second on a question of reachability — the model does not execute the program.

**Stage 8 — Deduplication.** The same defect appears many times: once per
scanner, once per path, once per call site. Collapse on the **defect identity**
— `(cwe, file, unit, sink_expression)` — not on the message text. Keep the
count: `occurrences` is signal about how exposed the defect is.

Match paths by parent directory plus filename tail. Deduplicating on a bare
basename silently merges two different files and loses a real finding.

**Stage 9 — Contextual verification.** For each surviving finding, look at the
surrounding code for the thing that makes it not-a-bug: a validator upstream, a
framework escaping the parameter, a type that cannot hold the payload, a caller
that only ever passes a constant. Record the verdict and the reason:
`confirmed`, `mitigated_by <what>`, or `needs_human`.

`needs_human` is a legitimate verdict and must stay available. A pipeline that
must decide will decide wrongly under uncertainty.

**Stage 10 — Feasibility filtering.** Drop what an attacker cannot actually
reach: code behind a feature flag that is off, an admin-only path in a service
with no admin, a sink whose input is fully constant. Record *why* each drop was
made, because the next scan will rediscover it and the reason is what stops
that work being repeated.

## Output contract

```json
{
  "findings": [
    {"id": "str", "cwe": "CWE-89", "file": "str", "line": 0, "unit": "str",
     "evidence": "str", "missing_control": "str",
     "occurrences": 1, "verdict": "confirmed|mitigated|needs_human",
     "verdict_reason": "str", "feasible": true, "confidence": 0.0}
  ],
  "dropped": [{"id": "str", "stage": 8, "why": "str"}],
  "counts": {"raw": 0, "deduped": 0, "verified": 0, "feasible": 0}
}
```

`counts` must be monotonically non-increasing across the four stages. If it is
not, the pipeline invented findings after the audit stage — stop and report.

## Failure modes

- **Confusing conformance with accuracy.** Output that matches this schema
  perfectly can still be entirely wrong. Schema validity is close to free;
  correctness is the expensive part. Never report conformance as a quality
  metric.
- **Dropping silently.** Every drop needs a stage and a reason.
- **Letting a model overrule dataflow on reachability.** It may propose a path;
  it may not confirm one.
- **Suppressing `needs_human` to look decisive.** Uncertainty that is hidden
  becomes someone's incident.

## Handoff

Feasible, confirmed findings go to **appsec-exploit-validate** for proof.
Everything else goes to **appsec-triage-report** with its verdict intact.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py
SCRIPT = "skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 5 · Where it breaks — deduplicating on the wrong key

The skill says to collapse on the **defect identity**, `(cwe, file, unit, sink_expression)`, and never on the message text. Here is why that sentence is in the procedure.

## 6 · The same failure, from a real model

Everything above is constructed. Here is the identical failure produced by an actual open-weight model — **Moonlight-16B-A3B**, Moonshot AI's MoE from the Kimi team — run on a Kaggle CPU kernel against this skill's output contract.

It was given the contract and two vulnerable functions: an `open()` on a caller-supplied path, and an `os.system()` on a caller-supplied argument. Its answer is reproduced verbatim below ([full run](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/kimi/moonlight-16b-completion-prompt.txt)).

## 7 · Read that output again

It passes the contract with zero problems, and almost nothing in it is true.

## 8 · The call graph, parsed rather than grepped

Three files of CyberTravels' booking service, parsed for real with `ast.parse`. The skill collects `FunctionDef` nodes and `Call` edges, marks the decorated handlers as entry points, walks the graph, and then splits the finding queue on the result.

Watch `jobs/runner.py`. It dispatches through `getattr(HANDLERS, name)()`, so the AST cannot say who calls anything in it — and every unreached function in that module lands in `unknown` rather than in the deletion list.

### The skill — [`skills/appsec/dead-code-ast-reachability/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dead-code-ast-reachability/SKILL.md)

```yaml
name: dead-code-ast-reachability
description: >-
  Build a call graph by parsing source to an abstract syntax tree, and use it to
  separate findings that are false positives about risk from findings that are
  false positives about code. Use when a queue is full of unreachable findings,
  when deciding whether a function is dead, or when a scanner reports a defect
  in code nothing calls.
allowed-tools: Read, Grep, Glob
```

# Dead code is a true positive about the code and a false positive about the risk

Both halves matter and teams act on only one.

The finding is **correct**: the concatenation is there, the sink is real, and a
reviewer who opens the file will agree. What is wrong is the implied
consequence, because nothing untrusted reaches it. Triaging it costs exactly as
much as triaging one on the login path, and there are usually far more of them —
so a queue that does not separate the two is a queue engineers learn to ignore,
which costs you the reachable ones as well.

Deciding which is which needs a **call graph**, and a call graph needs the
**abstract syntax tree**. Grep cannot do it: `def report` and `report(` and
`# report` are the same string to a regex, and a function named `run` appears in
every file in the repository. Parsing to an AST gives you the two node types the
question actually needs — `FunctionDef` for what exists, `Call` for what invokes
it — with their real nesting, so "which functions call this one" stops being a
text search and becomes a graph walk.

The AST is also honest about its own limits, which is the more useful half. It
resolves a literal call and it cannot resolve `getattr(mod, name)()`, a dispatch
dictionary, or a handler a framework wires by decorator at import time. Those
are not unreachable. They are **undecided**, and filing them as unreachable is
how a pipeline drops real bugs quietly.

## Step-by-step

**1 — Parse, do not grep.** `ast.parse(source)` per file. Collect every
`FunctionDef` and `AsyncFunctionDef` as a node; collect every `Call` as an edge
from its enclosing function.

**2 — Resolve each call to a name.** `Call.func` is a `Name` for `f()` and an
`Attribute` for `obj.f()`. Take `.id` or `.attr`. This is deliberately naive
about which `f` is meant — two functions of the same name in different modules
merge — and it errs toward *reachable*, which is the safe direction.

**3 — Mark entry points.** Anything decorated as a route or handler, anything
exported, anything a scheduler names. An entry point is reachable by definition,
and getting this list wrong is the largest source of error in the whole
procedure.

**4 — Walk from the entry points.** Everything reached is `reachable`.
Everything not reached is *either* dead *or* undecided, and step 5 decides.

**5 — Separate `unreachable` from `unknown`, never collapse them.** If the file
contains a dynamic call the AST could not resolve — `getattr`, a dispatch table,
a decorator that registers — every unreached function in that module is
`unknown`, not `unreachable`. Report the three buckets separately.

**6 — Only then classify the findings.** A finding on a reachable unit keeps its
severity. One on a genuinely dead unit is a false positive about risk and a true
positive about code, and belongs in a deletion list rather than a triage queue.
One on an `unknown` unit is unresolved work, not a clean result.

## Example

Input — two files, one entry point, one dynamic call:

```python
# api.py
@route("/reports")            # entry point
def list_reports(request):
    return render(load(request.args["id"]))

def load(rid): ...            # called by list_reports
def legacy_export(path): ...  # called by nothing

# jobs.py
def dispatch(name):
    return getattr(handlers, name)()   # unresolvable by AST
def nightly(): ...
```

Output:

```
reachable    list_reports, load           reached from an entry point
unreachable  legacy_export                no caller, no dynamic call in api.py
unknown      dispatch, nightly            jobs.py contains getattr(...)()
```

`nightly` is not dead. It is in a module the AST could not fully resolve, so
*every* unreached function in that module is undecided — and saying so is the
whole difference between this procedure and a shorter one.

## Output contract

```json
{
  "graph": {"functions": 0, "edges": 0, "entry_points": ["str"]},
  "buckets": {"reachable": 0, "unreachable": 0, "unknown": 0},
  "unresolved_calls": [{"file": "str", "why": "getattr|dispatch-table|decorator"}],
  "findings": [{"id": "str", "unit": "str", "bucket": "str",
                "true_about_code": true, "true_about_risk": false}],
  "queue": {"before": 0, "after": 0}
}
```

`unresolved_calls` is what makes the `unknown` bucket auditable. A report with
an empty `unknown` bucket and no `unresolved_calls` entry has either a very
simple codebase or a bug.

## Common edge cases

- **Two functions with the same name.** The naive resolver merges them and
  over-reports reachability. That is the safe direction; the unsafe one is a
  clever resolver that guesses wrong and marks a live function dead.
- **A decorator that registers the function.** `@app.route`, `@celery.task`,
  `@click.command`. The function has no caller in the source and is reachable
  from outside it. Treat every decorated function as an entry point unless you
  know the decorator.
- **Tests as callers.** A function called only from tests is dead in
  production. Exclude test files from the graph or you will never find any.
- **A method called through an instance.** `Attribute` gives you `.attr`, which
  is the method name without the class. Same over-reporting, same safe
  direction.
- **`__init__.py` re-exports.** A function imported and re-exported has no call
  edge at all and is reachable by import. Follow `ImportFrom` or accept it as
  `unknown`.

## Failure modes

- **Grepping for the name instead of parsing.** Comments, strings and unrelated
  functions with the same name all match, and the answer is unusable in either
  direction.
- **Collapsing `unknown` into `unreachable`.** It makes the queue shorter and
  it is how framework-wired handlers get dropped.
- **Trusting the entry-point list.** Every function is unreachable if you forgot
  the routes file.
- **Deleting on the graph's word alone.** Reachability from *untrusted* entry
  points is not reachability from anywhere; check both before a deletion.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dead-code-ast-reachability/scripts/dead_code_ast_reachability.py
SCRIPT = "skills/appsec/dead-code-ast-reachability/scripts/dead_code_ast_reachability.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 9 · The bucket that is work rather than a result

Read the last line of that output again: **2 reachable, 3 to delete, 1
unresolved.**

The three to delete are the easy win, and they are a deletion rather than a
triage — the finding goes because the code goes, and unlike a suppression
that cannot rot. The security queue turns out to be the cheapest to-delete list
in the building: already enumerated, already ranked by what each line would cost
if it ever became reachable again.

The unresolved one is the part that matters. `_settle` runs a SQL update and the
AST cannot tell you whether anything reaches it, because the module dispatches on
a runtime value. A two-bucket pipeline reports that as zero and calls the queue
clean. It is not clean; it is one finding nobody has looked at, filed under a
word that means *we did not check*.

One caution that does most of the remaining work: **"unreachable" and "dead" are
not synonyms.** A test fixture and a feature flag that has been off for two years
are both unreachable *under a condition*, and both become reachable the day
somebody changes one line. Only code with no caller anywhere, in a module the AST
fully resolved, is a deletion candidate.

## What you just proved

The call graph identifies three entry points, one of which uses dynamic dispatch. `load_report` is reachable, `debug_dump` and `legacy_export` are unknown rather than unreachable because runtime handler resolution cannot be ruled out. Two-bucket filtering silently drops both, and the three-bucket routing sends the unknowns to Phase 4 instead of paging or discarding them. The AST pass then parses three real files: 9 functions, 2 resolved call edges, 2 decorated entry points, and one `getattr(HANDLERS, name)()` that it records as unresolvable. That single unresolved call moves all three functions in `jobs/runner.py` into `unknown` — including `_settle`, which runs a SQL update. The six-finding queue splits 2 reachable, 3 to delete, 1 unresolved: every one is a true positive about the code, and three of them are false positives about the risk.

## Your turn

Two counts, and the second is the uncomfortable one. Count how many `unknown` cases your own reachability analysis produces and find out what your tooling does with them — if it reports them as clean, the number of real bugs you are dropping is the size of that bucket. Then open your suppression file and find the oldest entry. Check whether the code it covers is still unreachable, and whether anything in your pipeline would have told you if it stopped being.

---

**Next → [B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*